In [ ]:
import os
import json
import numpy as np
import torch
import cv2
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
from deeplsd.models.deeplsd_inference import DeepLSD
import line_understanding.feature_extraction as ft
from line_understanding.visualization import plot_images, plot_coplanar_lines, plot_lines_bool
import matplotlib.pyplot as plt

import torch.nn as nn
import torch_geometric.nn as pyg_nn
import sys
import os
sys.path.append(os.path.abspath("src"))  # Tells Python to treat "src" as package root



In [ ]:
import torch 


from lightning_tools.model import *

def run_inference_lightning(
    ckpt_path: str,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device = torch.device('cpu'),
    threshold_structural: float = 0.5,
    threshold_coplanarity: float = 0.5,
):
    # 1) load LightningModule (this also restores hparams)
    model = GNN.load_from_checkpoint(ckpt_path)
    model = model.to(device).eval()

    all_node_probs = []
    all_edge_probs = []
    all_node_preds = []
    all_edge_preds = []

    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            # 2) forward returns raw logits
            node_logits, edge_logits = model(batch)

            # 3) turn into probs
            node_probs = torch.sigmoid(node_logits)    # shape [B, N, 1] or [N,1]
            edge_probs = torch.sigmoid(edge_logits)    # shape [B, E, 1] or [E,1]


            all_node_probs.append(node_probs.cpu())
            all_edge_probs.append(edge_probs.cpu())
            # 4) threshold to get binary labels
            node_labels = (node_probs >= threshold_structural).float()
            edge_labels = (edge_probs >= threshold_coplanarity).float()

            all_node_preds.append(node_labels.cpu())
            all_edge_preds.append(edge_labels.cpu())

    return all_node_probs, all_node_preds, all_edge_probs, all_edge_preds


In [ ]:
import os
import json
import orjson

import numpy as np
import torch
import cv2
from torch_geometric.data import Data, Dataset
# from notebooks.models.dataset_utils import extract_line_feature_ROIAlign,sample_lines_grid
from sklearn.neighbors import NearestNeighbors

from typing import Optional, Tuple
import logging
    # -------------------------------------------



def line_geometry(line_pts: torch.Tensor, img_size: Tuple[int,int]):
    """
    line_pts : [N, 2, 2]  (x1,y1,x2,y2 per line)
    img_size : (H, W)
    returns   :  ϕ_node   [N, 5]
      [mid_x_norm, mid_y_norm, dir_x, dir_y, length_norm]
    """
    p1, p2   = line_pts[:, 0], line_pts[:, 1]           # [N,2]  [N,2]
    H, W     = img_size
    diag     = (H**2 + W**2)**0.5

    # 1) midpoint, then normalize to [0,1]
    mid      = 0.5 * (p1 + p2)                          # [N,2]
    mid_x    = mid[:,0:1] / W
    mid_y    = mid[:,1:2] / H

    # 2) unit direction
    vec      = p2 - p1
    dir_norm = F.normalize(vec, dim=1)                  # [N,2]

    # 3) length normalized by image diagonal
    length   = vec.norm(dim=1, keepdim=True) / diag     # [N,1]

    return torch.cat([mid_x, mid_y, dir_norm, length], dim=1)  # [N,5]
def _load_image(filepath: str, color_conversion: Optional[int] = None) -> Optional[np.ndarray]:
    """Loads an image using OpenCV."""
    if not os.path.exists(filepath):
        logging.error(f"Image file not found: {filepath}")
        return None
    try:
        img = cv2.imread(filepath, cv2.IMREAD_UNCHANGED) # Load as is (handles color, grayscale, alpha)
        if img is None:
            logging.error(f"Failed to load image (cv2.imread returned None): {filepath}")
            return None
        if color_conversion is not None:
            img = cv2.cvtColor(img, color_conversion)
        return img
    except Exception as e:
        logging.error(f"Error loading image {filepath}: {e}")
        return None



import os, json, logging
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from lightning_tools.line_sampler import LineSampler, EdgeSampler  # <-- your LightningModule
from lightning_tools.line_sampler import extract_line_feature_ROIAlign

class GraphDatasetInference(Dataset):
    def __init__(self, embeddings, image_path, coords, roi_output_size=(64, 64), method="sample", device=None, edge_sample_size = (32,16)):
        super().__init__()
                
        self.embeddings = embeddings
        self.image_path = image_path
        self.coords = coords
        self.roi_output_size = roi_output_size
        self.method = method
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.edge_sample_size = edge_sample_size

        num_samples_edge,width_edge = self.edge_sample_size

        self.edge_sampler = EdgeSampler(num_samples_u=num_samples_edge,num_samples_v=width_edge)

        # Only instantiate the sampler if we're going to use it
        if self.method == "sample":
            num_samples, width = self.roi_output_size
            self.sampler = LineSampler(
                num_samples=num_samples,
                width=width
            ).to(self.device)

    def __len__(self):
        return len(self.image_path)

    def __getitem__(self, idx):
        # load JSON
 
        file_path_img    = self.image_path[idx]

        # build node embeddings + labels
        feats, line_coords = self.embeddings[idx], self.coords[idx]
    

        x_emb = torch.tensor(np.vstack(np.array(feats)), dtype=torch.float)
        N     = x_emb.size(0)

        # === Feature extraction ===
        if self.method == "roi":
            roi_features = extract_line_feature_ROIAlign(
                img=_load_image(filepath=file_path_img, color_conversion=cv2.COLOR_BGR2RGB),
                lines=line_coords,
                output_size=self.roi_output_size,
                plot_results=True
            )
        else:  
            # 1) load & prep image tensor
            img_np = _load_image(filepath=file_path_img, color_conversion=cv2.COLOR_BGR2RGB)
            img_t  = (
                torch.tensor(img_np, dtype=torch.float32, device=self.device)
                     .div(255.0)
                     .permute(2, 0, 1)  # C,H,W
            )

            # 2) prep lines tensor
            lines_t = torch.tensor(line_coords, dtype=torch.float32, device=self.device)

            # 3) sample
            #    returns (N, C, num_samples, width)
            roi_features = self.sampler.sample_lines_grid(
                img=img_t,
                lines=lines_t,
                align_corners=True
            )

        # sanity‐check
        if roi_features is None or roi_features.shape[0] != N:
            logging.warning(
                f"ROI feature issue for {self.filter_json_files[idx]}; "
                f"got {None if roi_features is None else roi_features.shape}, expected ({N}, …)."
            )
            raise ValueError('ROI feature extraction failed.')

        # === build graph ===
        
        coords = torch.tensor(line_coords, dtype=torch.float)

 
        img_H, img_W = img_t.shape[-2:]                                  # after your ToTensor
        geo = line_geometry(coords, img_size=(img_H, img_W))


        # 2) helper to compute segment‐to‐segment distances
        def seg_seg_dist(p1, p2, q1, q2, eps=1e-8):
            # p1,p2: (N,2); q1,q2: (N,2) – here we compute N×N all-pairs
            # expand dims for broadcasting
            P1 = p1[:,None]  # (N,1,2)
            P2 = p2[:,None]  # (N,1,2)
            Q1 = q1[None,:]  # (1,N,2)
            Q2 = q2[None,:]  # (1,N,2)
            def proj(X, A, B):
                t = torch.clamp(((X-A)*(B-A)).sum(-1,keepdim=True) /
                                (((B-A)**2).sum(-1,keepdim=True)+eps), 0,1)
                return A + t*(B-A)
            # four point‐to‐segment cases
            d1 = ((Q1 - proj(Q1, P1, P2))**2).sum(-1)
            d2 = ((Q2 - proj(Q2, P1, P2))**2).sum(-1)
            d3 = ((P1 - proj(P1, Q1, Q2))**2).sum(-1)
            d4 = ((P2 - proj(P2, Q1, Q2))**2).sum(-1)
            return torch.sqrt(torch.min(torch.min(d1,d2), torch.min(d3,d4)))  # (N,N)


      
     
        full_edge_index, full_edge_labels = [], []

        for i in range(N):
            for j in range(N):  
                full_edge_index.append([i, j])

        
        full_edge_index = torch.tensor(full_edge_index, dtype=torch.long).t().contiguous()
        full_edge_labels = torch.tensor(full_edge_labels, dtype=torch.float).unsqueeze(1)
                
        img_t = torch.tensor(img_np, dtype=torch.float32).div(255).permute(2,0,1)
     

        
        p1, p2 = coords[:, 0], coords[:, 1]
        D = seg_seg_dist(p1, p2, p1, p2)  # (N,N)
        k = min(10, N - 1)
        k_global = min (50, N - 1)
        knn = D.topk(k+1, largest=False).indices[:, 1:]  # (N, k)
        knn_global = D.topk(k_global+1, largest=False).indices[:, 1:]  # (N, k)

        # 11a) local edges:
        src = torch.arange(N, device=coords.device).unsqueeze(1).expand(-1, k).reshape(-1)
        dst = knn.reshape(-1)
        local_edge_index = torch.stack([src, dst], dim=0)  # (2, N*k)
        
        # 11a) local edges:
        src_global = torch.arange(N, device=coords.device).unsqueeze(1).expand(-1, k_global).reshape(-1)
        dst_global = knn_global.reshape(-1)
        global_edge_index = torch.stack([src_global, dst_global], dim=0)  # (2, N*k)

        lines_i = coords[src_global]  # (N*k, 2,2)
        lines_j = coords[dst_global]  # (N*k, 2,2)
        quads = torch.stack([
            lines_i[:, 0],  # start_i
            lines_i[:, 1],  # end_i
            lines_j[:, 1],  # end_j
            lines_j[:, 0],  # start_j
        ], dim=1)          # (N*k, 4,2)

        # 3) now call the sampler and overwrite both the edge_attr and edge_index
        edge_attr = self.edge_sampler(img_t, quads)

        return Data(
            x=x_emb,
            coordinates=coords,
            geo=geo,
            edge_index=local_edge_index,
            global_edge_index = global_edge_index,
            full_edge_index=full_edge_index,
            full_edge_labels=full_edge_labels,
            roi_features=roi_features,
            edge_attr = edge_attr,
            
        )



In [ ]:
# Paramaeters
image_pth          = 'data/color.jpg'

ckpt_pth = "lightning_tools/checkpoints/model.ckpt"
device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
color_img = cv2.imread(image_pth) # Load as is (handles color, grayscale, alpha)

color_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB)

gray_img = cv2.cvtColor(color_img, cv2.COLOR_RGB2GRAY)

plot_images([color_img], ["Image"])

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conf = {'detect_lines': True, 'line_detection_params': {'merge': False, 'filtering': True, 'grad_thresh': 3}}
ckpt = torch.load('../weights/deeplsd_md.tar', map_location=device, weights_only=False)
net = DeepLSD(conf)
net.load_state_dict(ckpt['model'])
net = net.to(device).eval()
    

In [ ]:
df_intermediate_features = None
angle_intermediate_features = None 
df_hook_handle = net.df_head[5].register_forward_hook(ft.hook_df)
angle_hook_handle = net.angle_head[5].register_forward_hook(ft.hook_angle)
input_tensor = torch.tensor(gray_img, dtype=torch.float32, device=device)[None, None] / 255.
with torch.no_grad():
    out = net({'image': input_tensor})
    pred_lines = out['lines'][0]
    if isinstance(pred_lines, torch.Tensor):
        pred_lines = pred_lines.cpu().numpy()
        
# get embeddings for intermediate layers.
combined_features = torch.cat([ft.df_intermediate_features, ft.angle_intermediate_features], dim=1)
downsample_ratio = color_img.shape[1] / combined_features.shape[3]
df_hook_handle.remove()
angle_hook_handle.remove()

In [ ]:
embeddings = []
coordinates = []
for i, l in enumerate(pred_lines):
    line = l.reshape(2, 2) if l.shape == (4,) else l
    coordinates.append(l.tolist())
    line_embedding = ft.sample_line_features(combined_features, line, num_samples=10, downsample_ratio=downsample_ratio).tolist()
    embeddings.append(line_embedding)

In [ ]:
dataset = GraphDatasetInference([embeddings], [image_pth], [coordinates], roi_output_size=[32, 32], edge_sample_size=[8,8])


In [ ]:
sample = dataset[0]
rf = sample.roi_features               # shape: (N, C, H, W)
flat_rf = rf.view(rf.size(0), -1)      # → (N,  C*H*W )
print("dataset roi_features shape:", rf.shape)
print(" flattened ROI dim:", flat_rf.size(1))



In [ ]:
# node_preds, edge_preds = run_inference(model, dataset, model_path=model_pth, device=device)
node_probs, node_preds, edge_probs, edge_preds = run_inference_lightning(ckpt_path=ckpt_pth, data_loader=dataset, device=device, threshold_structural=0.5, threshold_coplanarity=0.74)


In [ ]:
fig, ax = plt.subplots()

plot_lines_bool(ax, color_img, pred_lines, node_preds[0].flatten().tolist())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt

def plot_lines_scores(
    ax, img, lines, scores,
    color='red',
    alpha_min=0,
    alpha_max=1.0
):
    """
    Plot lines with transparency proportional to their score.
    
    Parameters
    ----------
    ax : matplotlib Axes
    img : (H,W) or (H,W,3) image array
    lines : list of [(x0,y0), (x1,y1)] pairs
    scores : list or array of floats (any range)
    color : line color
    alpha_min : minimum alpha (for lowest score)
    alpha_max : maximum alpha (for highest score)
    """
    scores = np.asarray(scores, dtype=float)
    # normalize to [0,1]
    if scores.max() > scores.min():
        norm = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        norm = np.ones_like(scores)
    # map to [alpha_min, alpha_max]
    alphas = alpha_min + norm * (alpha_max - alpha_min)

    for l, a in zip(lines, alphas):
        line = plt.Line2D(
            (l[0][0], l[1][0]),
            (l[0][1], l[1][1]),
            linewidth=2,
            color=color,
            alpha=a
        )
        ax.add_line(line)

    ax.imshow(img, cmap='gray')
    ax.set_axis_off()


fig, ax = plt.subplots()

plot_lines_scores(ax, color_img, pred_lines, node_probs[0].flatten().tolist())

In [ ]:

# Convert to NumPy
edge_preds_array = edge_preds[0].numpy()
print(edge_preds_array.shape)

edge_preds_array = edge_preds_array.reshape((len(pred_lines), -1))
print(edge_preds_array.shape)

edge_probs_array = edge_probs[0].numpy()
print(edge_preds_array.shape)

edge_probs_array = edge_probs_array.reshape((len(pred_lines), -1))
print(edge_preds_array.shape)

In [ ]:
import numpy as np
import hdbscan
from sklearn.metrics import silhouette_score

def cluster_lines_hdbscan(distance_matrix: np.ndarray,
                          min_cluster_size: int = 2,
                          min_samples: int | None = None,
                          scan_min_size: bool = False,
                          size_grid: int = 10,
                          verbose: bool = False):
    """
    Cluster an NxN coplanarity-distance matrix using HDBSCAN.

    Parameters
    ----------
    distance_matrix : (N, N) ndarray
        Symmetric, zero diagonal.
    min_cluster_size : int, default 2
        The minimum size of clusters; passed to HDBSCAN.
    min_samples : int | None, default None
        The number of samples in a neighborhood for a point to be considered
        a core point. If None, uses the same value as min_cluster_size.
    scan_min_size : bool, default False
        If True, will try `size_grid` different min_cluster_size values between
        `min_cluster_size` and ⌈√N⌉+1, and pick the one maximizing silhouette.
    size_grid : int, default 10
        Number of min_cluster_size values to try if scan_min_size is True.
    verbose : bool, default False
        Print silhouette scores for each tried size.

    Returns
    -------
    labels : (N,) ndarray of int
        Cluster labels (noise = -1).
    best_size : int
        The min_cluster_size that was used.
    best_silhouette : float | None
        The silhouette score (None if all points noise or only one cluster).
    """
    D = np.asarray(distance_matrix, dtype=float)
    if D.shape[0] != D.shape[1]:
        raise ValueError("Distance matrix must be square")
    N = D.shape[0]

    # helper to fit & score
    def fit_and_score(mcs):
        model = hdbscan.HDBSCAN(
            metric='precomputed',
            min_cluster_size=mcs,
            min_samples=min_samples or mcs
        )
        labels = model.fit_predict(D)
        # need at least 2 non-noise clusters for silhouette
        if len(set(labels) - {-1}) < 2:
            return labels, None
        score = silhouette_score(D, labels, metric='precomputed')
        return labels, score

    # if not scanning, just do one run
    if not scan_min_size:
        labels, score = fit_and_score(min_cluster_size)
        return labels, min_cluster_size, score

    # scan over a grid of sizes
    max_size = int(np.sqrt(N)) + 1
    sizes = np.unique(
        np.linspace(min_cluster_size, max_size, size_grid, dtype=int)
    )
    best_score = -1.0
    best = (None, None)
    for mcs in sizes:
        labels, score = fit_and_score(mcs)
        if verbose:
            print(f"min_cluster_size={mcs}, silhouette={score}")
        if score is not None and score > best_score:
            best_score = score
            best = (labels, mcs)
    if best[0] is None:
        raise RuntimeError("HDBSCAN found fewer than 2 clusters for all tried sizes.")
    return best[0], best[1], best_score


    

In [ ]:
D = 1 - edge_probs_array
np.fill_diagonal(D, 0)


labels, _, _ = cluster_lines_hdbscan(D, min_cluster_size=3, min_samples=2, size_grid=10000)


fig, ax = plt.subplots(figsize=(8,6))
plot_coplanar_lines(
    ax,
    pred_lines,      
    labels,      # the (N,) array returned by cluster_lines
    color_img        # the background image array (H×W×3 or H×W)
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

# Parameters
batch_size = 4  # Number of subplots per figure
num_items = len(edge_preds_array)
num_batches = math.ceil(num_items / batch_size)

for batch in range(num_batches):
    start_idx = batch * batch_size
    end_idx = min(start_idx + batch_size, num_items)
    current_batch_size = end_idx - start_idx

    # Create subplots for this batch
    fig, axes = plt.subplots(nrows=1, ncols=current_batch_size, figsize=(4 * current_batch_size, 4))
    
    # Ensure axes is iterable
    if current_batch_size == 1:
        axes = [axes]
    
    for i, ax in enumerate(axes):
        idx = start_idx + i
        plot_coplanar_lines_full(ax, pred_lines, edge_preds_array[idx], color_img, idx)
        ax.set_title(f'Coplanarity of line {idx + 1}')
    
    plt.tight_layout()
    plt.show()

